# J4 · Évaluation du pré-remplissage sur le jeu annoté par des humains

1. Envoie chaque photo **utilisable** à l'API StreamSentinel (le notebook `API_streamsentinel` doit tourner dans un autre onglet).
2. Compare les réponses de l'IA à celles des annotateurs, question par question.
3. Calcule : **taux de pré-remplissage**, **précision** avec intervalle de confiance (Wilson), **calibration** de la confiance,
   **accord entre annotateurs** (kappa de Cohen), et les confusions à surveiller (écume / mousse, déchets).

Règles d'évaluation :
- une question n'est évaluée que si l'humain a donné une réponse (pas « ? ») ;
- sur les photos communes, si les deux annotateurs ne sont pas d'accord, la question est **écartée** pour cette photo ;
- quand le sens de la photo est inconnu, l'IA reçoit « aval » par défaut. Les réponses humaines sur les rives
  n'existent alors que si les deux berges sont identiques, donc la comparaison reste juste.

In [ ]:
URL_API = "https://directed-freight-gully.ngrok-free.dev"   # adresse fixe ngrok
ANNOTATEURS = ["miryame", "frere"]

import os, csv, json, time, math, random, requests
from collections import Counter, defaultdict
from google.colab import drive
drive.mount('/content/drive')

DOSSIER = "/content/drive/MyDrive/streamsentinel/eval_j4"
CACHE = f"{DOSSIER}/resultats_ia.json"
ENTETES = {"ngrok-skip-browser-warning": "true"}
print("API :", requests.get(f"{URL_API}/sante", headers=ENTETES, timeout=15).json())


Mounted at /content/drive
API : {'ok': True, 'gpu': True, 'analyses_en_cours': 0}


## 1. Annotations humaines

In [ ]:
def lire(nom):
    chemin = f"{DOSSIER}/annotations_{nom}.csv"
    if not os.path.exists(chemin):
        return {}
    with open(chemin, encoding="utf-8") as f:
        return {l["fichier"]: l for l in csv.DictReader(f)}

annot = {nom: lire(nom) for nom in ANNOTATEURS}
for nom, a in annot.items():
    print(f"{nom:8s} : {len(a)} photos annotées, {sum(l['utilisable'] == 'oui' for l in a.values())} utilisables")

QUESTIONS = ["channel_form", "bottom_type", "bank_type", "water_flow", "water_aspect", "barriers",
             "draining_pipes", "sewage_discharge", "construction", "impervious_left", "impervious_right",
             "vegetation_left", "vegetation_right", "vegetation_type_left", "vegetation_type_right",
             "vegetation_cuts", "overall", "dechets_visibles"]

# Référence humaine : une photo est gardée si au moins un annotateur la juge utilisable.
# Sur les photos communes, une question n'est gardée que si les deux sont d'accord.
reference, orientation = {}, {}
for photo in sorted(set().union(*[set(a) for a in annot.values()])):
    lignes = [a[photo] for a in annot.values() if photo in a]
    if not any(l["utilisable"] == "oui" for l in lignes):
        continue
    lignes = [l for l in lignes if l["utilisable"] == "oui"]
    ref = {}
    for q in QUESTIONS:
        valeurs = {l[q] for l in lignes if l.get(q, "?") != "?"}
        ref[q] = valeurs.pop() if len(valeurs) == 1 and all(l.get(q, "?") != "?" for l in lignes) else "?"
    reference[photo] = ref
    sens = {l["orientation"] for l in lignes}
    orientation[photo] = sens.pop() if len(sens) == 1 else "?"
print(len(reference), "photos utilisables dans la référence")


miryame  : 42 photos annotées, 21 utilisables
frere    : 0 photos annotées, 0 utilisables
21 photos utilisables dans la référence


## 2. Réponses de l'IA (via l'API, avec cache)

Compter environ 45 s par photo. Les résultats sont gardés dans `resultats_ia.json` : relancer la cellule ne refait que les photos manquantes.

In [ ]:
cache = json.load(open(CACHE)) if os.path.exists(CACHE) else {}

def analyser(photo, sens):
    with open(f"{DOSSIER}/photos/{photo}", "rb") as f:
        rep = requests.post(f"{URL_API}/analyser", headers=ENTETES, timeout=120,
                            files={"photo": (photo, f, "image/jpeg")},
                            data={"orientation": sens if sens in ("amont", "aval") else "aval"})
    ident = rep.json()["id"]
    for _ in range(200):
        time.sleep(4)
        t = requests.get(f"{URL_API}/resultat/{ident}", headers=ENTETES, timeout=60).json()
        if t.get("etat") == "termine":
            res = t["resultat"]
            res.pop("image_annotee_b64", None)   # inutile pour l'évaluation, et lourd
            return res
        if t.get("etat") == "erreur":
            raise RuntimeError(t.get("erreur"))
    raise TimeoutError(photo)

for i, photo in enumerate(reference):
    if photo in cache:
        continue
    debut = time.time()
    try:
        cache[photo] = analyser(photo, orientation[photo])
    except Exception as e:
        print(f"⚠️ {photo} : {e}")
        continue
    json.dump(cache, open(CACHE, "w"), ensure_ascii=False)
    print(f"{i + 1}/{len(reference)} {photo[:50]} ({time.time() - debut:.0f} s)")
print(len(cache), "photos analysées par l'IA")


21 photos analysées par l'IA


## 3. Comparaison

In [ ]:
def reponse_ia(res, q):
    fiche = res.get("fiche", {})
    if not fiche.get("valide", True):
        return "NOT_SURE", 0.0
    if q == "overall":
        oa = fiche.get("overall_assessment", {})
        return oa.get("valeur", "NOT_SURE"), oa.get("confiance", 0.0)
    if q == "dechets_visibles":
        d = fiche.get("signalements", {}).get("dechets")
        return ("NOT_SURE", 0.0) if d is None else ("YES" if len(d) else "NO", 1.0)
    r = fiche.get("formulaire", {}).get(q, {})
    return r.get("valeur", "NOT_SURE"), r.get("confiance", 0.0)

lignes = []   # une ligne par (photo, question) où l'humain a répondu
for photo, ref in reference.items():
    if photo not in cache:
        continue
    for q in QUESTIONS:
        if ref[q] == "?":
            continue
        ia, conf = reponse_ia(cache[photo], q)
        lignes.append({"photo": photo, "question": q, "humain": ref[q], "ia": ia, "confiance": conf,
                       "repondu": ia not in (None, "NOT_SURE"), "juste": ia == ref[q]})
print(len(lignes), "paires question / réponse humaine")


239 paires question / réponse humaine


## 4. Résultats

- **Pré-remplissage** : part des questions (connues de l'humain) auxquelles l'IA ose répondre.
- **Précision** : part de réponses justes **parmi celles données par l'IA**, avec intervalle de Wilson à 95 %.
- **Global** : intervalle par *bootstrap sur les photos* (les questions d'une même photo ne sont pas indépendantes).

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan")
    p = k / n
    centre, marge = (p + z * z / (2 * n)) / (1 + z * z / n), z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return max(0, centre - marge), min(1, centre + marge)

print(f"{'Question':24s} {'n':>3s} {'pré-rempli':>10s} {'précision':>9s}  IC 95 %")
tableau = []
for q in QUESTIONS:
    L = [l for l in lignes if l["question"] == q]
    rep = [l for l in L if l["repondu"]]
    k = sum(l["juste"] for l in rep)
    bas, haut = wilson(k, len(rep))
    tableau.append((q, len(L), len(rep) / len(L) if L else float("nan"), k / len(rep) if rep else float("nan"), bas, haut))
    print(f"{q:24s} {len(L):3d} {tableau[-1][2]:10.0%} {tableau[-1][3]:9.0%}  [{bas:.0%} ; {haut:.0%}]" if rep else f"{q:24s} {len(L):3d}  aucune réponse de l'IA")

def globaux(sous_lignes):
    rep = [l for l in sous_lignes if l["repondu"]]
    return len(rep) / max(len(sous_lignes), 1), sum(l["juste"] for l in rep) / max(len(rep), 1)

par_photo = defaultdict(list)
for l in lignes:
    par_photo[l["photo"]].append(l)
photos = list(par_photo)
random.seed(0)
tirages = sorted((globaux([l for p in random.choices(photos, k=len(photos)) for l in par_photo[p]]) for _ in range(1000)))
couverture, precision = globaux(lignes)
cs, ps = sorted(t[0] for t in tirages), sorted(t[1] for t in tirages)
print(f"\nGLOBAL sur {len(photos)} photos : pré-rempli {couverture:.0%} [{cs[25]:.0%} ; {cs[974]:.0%}]"
      f"  ·  précision {precision:.0%} [{ps[25]:.0%} ; {ps[974]:.0%}]")


Question                   n pré-rempli précision  IC 95 %
channel_form              12        83%       50%  [24% ; 76%]
bottom_type               10        70%       86%  [49% ; 97%]
bank_type                 16        81%      100%  [77% ; 100%]
water_flow                19        79%       73%  [48% ; 89%]
water_aspect              10        90%       89%  [56% ; 98%]
barriers                  19        63%       75%  [47% ; 91%]
draining_pipes            10        10%        0%  [0% ; 79%]
sewage_discharge           8  aucune réponse de l'IA
construction              18        11%      100%  [34% ; 100%]
impervious_left           13        69%       67%  [35% ; 88%]
impervious_right          13        69%       78%  [45% ; 94%]
vegetation_left           11        91%       70%  [40% ; 89%]
vegetation_right          11        82%       56%  [27% ; 81%]
vegetation_type_left      10        60%       83%  [44% ; 97%]
vegetation_type_right      9        67%       50%  [19% ; 81%]
veget

## 5. Calibration : la confiance affichée correspond-elle au taux de réussite ?

In [ ]:
tranches = [(0, 0.5), (0.5, 0.7), (0.7, 0.9), (0.9, 1.01)]
print(f"{'Confiance IA':14s} {'n':>4s} {'réussite':>9s}")
for bas, haut in tranches:
    T = [l for l in lignes if l["repondu"] and bas <= (l["confiance"] or 0) < haut]
    if T:
        k = sum(l["juste"] for l in T)
        b, h = wilson(k, len(T))
        print(f"{bas:.1f} – {min(haut, 1):.1f}      {len(T):4d} {k / len(T):9.0%}  [{b:.0%} ; {h:.0%}]")
print("Bien calibré = la réussite monte avec la confiance, et reste proche de la confiance affichée.")


Confiance IA      n  réussite
0.5 – 0.7         9       44%  [19% ; 73%]
0.7 – 0.9        86       63%  [52% ; 72%]
0.9 – 1.0        50       90%  [79% ; 96%]
Bien calibré = la réussite monte avec la confiance, et reste proche de la confiance affichée.


## 6. Confusions à surveiller

In [ ]:
def confusion(q):
    c = Counter((l["humain"], l["ia"]) for l in lignes if l["question"] == q)
    print(f"\n{q} (humain → IA)")
    for (h, i), n in sorted(c.items(), key=lambda x: -x[1]):
        print(f"  {h:14s} → {i:14s} {n}" + ("" if h == i else "   ✗" if i != "NOT_SURE" else "   (abstention)"))

for q in ("water_aspect", "barriers", "dechets_visibles", "overall"):
    confusion(q)

print("\nErreurs sur l'aspect de l'eau (à regarder une par une) :")
for l in lignes:
    if l["question"] == "water_aspect" and l["repondu"] and not l["juste"]:
        print(f"  {l['photo'][:55]:55s} humain {l['humain']:8s} IA {l['ia']}")



water_aspect (humain → IA)
  CLEAR          → CLEAR          5
  MUDDY          → MUDDY          2
  ALTERED_COLOR  → ALTERED_COLOR  1
  FOAM           → CLEAR          1   ✗
  FOAM           → NOT_SURE       1   (abstention)

barriers (humain → IA)
  NO             → NO             8
  NO             → NOT_SURE       7   (abstention)
  YES            → NO             2   ✗
  YES            → YES            1
  NO             → YES            1   ✗

dechets_visibles (humain → IA)
  NO             → NO             10
  NO             → NOT_SURE       3   (abstention)
  NO             → YES            1   ✗
  YES            → NO             1   ✗

overall (humain → IA)
  GOOD           → GOOD           4
  MODERATE       → POOR           3   ✗
  MODERATE       → GOOD           3   ✗
  MODERATE       → MODERATE       2
  GOOD           → NOT_SURE       2   (abstention)
  GOOD           → MODERATE       1   ✗
  MODERATE       → NOT_SURE       1   (abstention)
  POOR           → MODERATE  

## 7. Accord entre annotateurs (photos communes)

Le kappa de Cohen corrige l'accord dû au hasard : > 0,6 bon, > 0,8 très bon. C'est la **référence de difficulté** : on ne peut pas exiger de l'IA plus d'accord qu'entre deux humains.

In [ ]:
def kappa(paires):
    n = len(paires)
    if n == 0:
        return float("nan"), float("nan"), 0
    accord = sum(a == b for a, b in paires) / n
    ca, cb = Counter(a for a, _ in paires), Counter(b for _, b in paires)
    hasard = sum(ca[k] * cb[k] for k in set(ca) | set(cb)) / (n * n)
    return accord, (accord - hasard) / (1 - hasard) if hasard < 1 else 1.0, n

a1, a2 = (annot[n] for n in ANNOTATEURS)
communes = [p for p in a1 if p in a2 and a1[p]["utilisable"] == a2[p]["utilisable"] == "oui"]
print(f"{len(communes)} photos communes utilisables\n{'Question':24s} {'n':>3s} {'accord':>7s} {'kappa':>6s}")
toutes = []
for q in QUESTIONS:
    paires = [(a1[p][q], a2[p][q]) for p in communes if a1[p][q] != "?" and a2[p][q] != "?"]
    toutes += paires
    acc, k, n = kappa(paires)
    if n:
        print(f"{q:24s} {n:3d} {acc:7.0%} {k:6.2f}")
acc, k, n = kappa(toutes)
print(f"{'TOUTES QUESTIONS':24s} {n:3d} {acc:7.0%} {k:6.2f}" if n else "Pas encore d'annotations communes du second annotateur.")


0 photos communes utilisables
Question                   n  accord  kappa
Pas encore d'annotations communes du second annotateur.


## 8. Sauvegarde du tableau (pour le README)

In [ ]:
with open(f"{DOSSIER}/evaluation_j4.md", "w", encoding="utf-8") as f:
    f.write(f"Human-annotated evaluation set: {len(photos)} usable photos (Wikimedia Commons, CC licences).\n\n")
    f.write("| Question | n | Pre-filled | Accuracy | 95% CI |\n|---|---|---|---|---|\n")
    for q, n, cov, acc, b, h in tableau:
        f.write(f"| {q} | {n} | {cov:.0%} | " + (f"{acc:.0%} | {b:.0%}–{h:.0%} |\n" if acc == acc else "– | – |\n"))
    f.write(f"\n**Overall**: pre-filled {couverture:.0%} [{cs[25]:.0%}–{cs[974]:.0%}], "
            f"accuracy {precision:.0%} [{ps[25]:.0%}–{ps[974]:.0%}] (bootstrap over photos).\n")
print(open(f"{DOSSIER}/evaluation_j4.md", encoding="utf-8").read())


Human-annotated evaluation set: 21 usable photos (Wikimedia Commons, CC licences).

| Question | n | Pre-filled | Accuracy | 95% CI |
|---|---|---|---|---|
| channel_form | 12 | 83% | 50% | 24%–76% |
| bottom_type | 10 | 70% | 86% | 49%–97% |
| bank_type | 16 | 81% | 100% | 77%–100% |
| water_flow | 19 | 79% | 73% | 48%–89% |
| water_aspect | 10 | 90% | 89% | 56%–98% |
| barriers | 19 | 63% | 75% | 47%–91% |
| draining_pipes | 10 | 10% | 0% | 0%–79% |
| sewage_discharge | 8 | 0% | – | – |
| construction | 18 | 11% | 100% | 34%–100% |
| impervious_left | 13 | 69% | 67% | 35%–88% |
| impervious_right | 13 | 69% | 78% | 45%–94% |
| vegetation_left | 11 | 91% | 70% | 40%–89% |
| vegetation_right | 11 | 82% | 56% | 27%–81% |
| vegetation_type_left | 10 | 60% | 83% | 44%–97% |
| vegetation_type_right | 9 | 67% | 50% | 19%–81% |
| vegetation_cuts | 17 | 6% | 0% | 0%–79% |
| overall | 18 | 78% | 43% | 21%–67% |
| dechets_visibles | 15 | 80% | 83% | 55%–95% |

**Overall**: pre-filled 61% [45%–7